In [ ]:
!pip install nibabel

In [ ]:
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os

path = "/content/drive/MyDrive/ADNI_Project/MRI_Images"

print(os.listdir(path))

In [ ]:
import os

base_path = "/content/drive/MyDrive/ADNI_Project/MRI_Images"

nii_files = []

for root, dirs, files in os.walk(base_path):
    for file in files:
        if file.endswith(".nii"):
            nii_files.append(os.path.join(root, file))

print("Total MRI files:", len(nii_files))

In [ ]:
import os

base_path = "/content/drive/MyDrive/ADNI_Project/MRI_Images"

for folder in sorted(os.listdir(base_path)):
    folder_path = os.path.join(base_path, folder)
    if os.path.isdir(folder_path):
        count = 0
        for root, dirs, files in os.walk(folder_path):
            count += sum(file.endswith(".nii") for file in files)
        print(folder, ":", count)

In [ ]:
import os
import pandas as pd

base_path = "/content/drive/MyDrive/ADNI_Project/MRI_Images"

mri_records = []

for root, dirs, files in os.walk(base_path):
    for file in files:

        if file.endswith(".nii"):

            image_id = file.split("_")[-1].replace(".nii", "")

            mri_records.append(
                [image_id,
                 os.path.join(root, file)]
            )

mri_inventory = pd.DataFrame(
    mri_records,
    columns=["IMAGE_DATA_ID", "MRI_PATH"]
)

print(mri_inventory.shape)

mri_inventory.head()

In [ ]:
print(
    mri_inventory["IMAGE_DATA_ID"].head()
)

In [ ]:
# Load final MRI metadata for the 702 matched participants
final_mri_metadata = pd.read_csv(
    "/content/drive/MyDrive/ADNI_Project/Cognitive_Data/final_mri_metadata_702.csv"
)

# Rename Image Data ID column to match inventory column name
final_mri_metadata = final_mri_metadata.rename(
    columns={"Image Data ID": "IMAGE_DATA_ID"}
)

# Convert to string to avoid matching issues
final_mri_metadata["IMAGE_DATA_ID"] = final_mri_metadata["IMAGE_DATA_ID"].astype(str)
mri_inventory["IMAGE_DATA_ID"] = mri_inventory["IMAGE_DATA_ID"].astype(str)

# Match downloaded MRI files with metadata
matched_mri_files = final_mri_metadata.merge(
    mri_inventory,
    on="IMAGE_DATA_ID",
    how="inner"
)

print("Matched MRI files:", matched_mri_files.shape)
print("Unique RIDs:", matched_mri_files["RID"].nunique())
print("Unique Image IDs:", matched_mri_files["IMAGE_DATA_ID"].nunique())

matched_mri_files.head()

In [ ]:
# Sort by RID and acquisition date
matched_mri_files["Acq Date"] = pd.to_datetime(matched_mri_files["Acq Date"])

matched_mri_unique = (
    matched_mri_files
    .sort_values(["RID", "Acq Date"])
    .drop_duplicates(subset=["RID"], keep="first")
)

print("Final MRI files:", matched_mri_unique.shape)
print("Unique RIDs:", matched_mri_unique["RID"].nunique())
print("Unique Image IDs:", matched_mri_unique["IMAGE_DATA_ID"].nunique())

In [ ]:
matched_mri_unique.to_csv(
    "/content/drive/MyDrive/ADNI_Project/Cognitive_Data/final_mri_files_702.csv",
    index=False
)

In [ ]:
import nibabel as nib

sample_path = matched_mri_unique["MRI_PATH"].iloc[0]

img = nib.load(sample_path)

print("Shape:", img.shape)

In [ ]:
data = img.get_fdata()

print("Min:", data.min())
print("Max:", data.max())
print("Mean:", data.mean())

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,6))

plt.imshow(
    data[:, :, data.shape[2] // 2],
    cmap="gray"
)

plt.axis("off")
plt.show()

In [ ]:
import nibabel as nib
from collections import Counter

shapes = []

for path in matched_mri_unique["MRI_PATH"]:

    img = nib.load(path)

    shapes.append(img.shape)

shape_counts = Counter(shapes)

print("Number of unique shapes:", len(shape_counts))
print("\nMost common shapes:")

for shape, count in shape_counts.most_common(10):
    print(shape, ":", count)

In [ ]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

In [ ]:
import os
import nibabel as nib
import numpy as np
from skimage.transform import resize
from tqdm import tqdm

TARGET_SHAPE = (160, 160, 160)

output_dir = "/content/drive/MyDrive/ADNI_Project/MRI_Images/Processed_MRI_160"
os.makedirs(output_dir, exist_ok=True)

for index, row in tqdm(matched_mri_unique.iterrows(), total=len(matched_mri_unique)):

    rid = row["RID"]
    image_id = row["IMAGE_DATA_ID"]
    path = row["MRI_PATH"]

    img = nib.load(path)
    data = img.get_fdata()

    data_resized = resize(
        data,
        TARGET_SHAPE,
        mode="constant",
        anti_aliasing=True
    )

    data_normalized = (
        data_resized - data_resized.min()
    ) / (
        data_resized.max() - data_resized.min()
    )

    save_path = os.path.join(
        output_dir,
        f"RID_{rid}_IMG_{image_id}.npy"
    )

    np.save(save_path, data_normalized.astype(np.float32))

print("MRI preprocessing completed.")

In [ ]:
import os

output_dir = "/content/drive/MyDrive/ADNI_Project/MRI_Images/Processed_MRI_160"

files = [f for f in os.listdir(output_dir) if f.endswith(".npy")]

print("Total processed MRI files:", len(files))
print(files[:5])

In [ ]:
import numpy as np

sample = np.load(
    "/content/drive/MyDrive/ADNI_Project/MRI_Images/Processed_MRI_160/" + files[0]
)

print("Shape:", sample.shape)
print("Min:", sample.min())
print("Max:", sample.max())